In [1]:
from pymongo import MongoClient
# Set up the Jupyter version of Dash
from dash import Dash, dcc, html, dash_table, callback_context

# Configure the necessary Python module imports
import dash_leaflet as dl
import plotly.express as px
import plotly.graph_objects as go
from dash.dependencies import Input, Output, State
import base64
import pandas as pd
import numpy as np
from datetime import datetime

# Import your CRUD Python module
from animal_shelter import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Update with your credentials and connection details
client = MongoClient("mongodb://localhost:27017")
print(client.list_database_names())
username = 'aacuser'
password = '9un56f38t'
host = 'localhost'
port = 27017
db = 'AAC'
collection = 'animals'

# Create an instance of your CRUD module
shelter = AnimalShelter(username, password, host, port, db, collection)

# Class read method must support return of list object and accept projection json input
# Initially fetch all data for unfiltered view
df = pd.DataFrame.from_records(shelter.read({}))

# Remove the _id column to prevent DataTable issues
if '_id' in df.columns:
    df.drop(columns=['_id'], inplace=True)

# TODO: Update path for the correct image
# Load the Grazioso Salvare logo
image_filename = 'images/PlaceholderImage.jpg'
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode('ascii')

#########################
# Dashboard Layout / View
#########################
app = Dash('Grazioso_Salvare_Dashboard')

# Define the filter options based on the Rescue Type and Preferred Dog Breeds Table
app.layout = html.Div([
    # Header with logo and title
    html.Div([
        html.Div([
            html.A(
                html.Img(
                    src=f'data:image/png;base64,{encoded_image}',
                    style={'height': '100px', 'margin-right': '15px'}
                ),
                href='https://www.snhu.edu',  # Link to client's homepage as requested
                target='_blank'
            )
        ], style={'display': 'inline-block', 'vertical-align': 'middle'}),

        html.Div([
            html.H1('Grazioso Salvare - Rescue Animal Dashboard'),
            html.H3('Ellis Miller')
        ], style={'display': 'inline-block', 'vertical-align': 'middle'})
    ], style={'padding': '10px', 'background-color': '#f2f2f2'}),

    html.Hr(),

    # Filter options
    html.Div([
        html.H3('Filter by Rescue Type:'),
        dcc.RadioItems(
            id='filter-type',
            options=[
                {'label': 'Water Rescue', 'value': 'water'},
                {'label': 'Mountain or Wilderness Rescue', 'value': 'mountain'},
                {'label': 'Disaster or Individual Tracking', 'value': 'disaster'},
                {'label': 'Reset (All)', 'value': 'all'}
            ],
            value='all',
            labelStyle={'display': 'inline-block', 'margin-right': '10px', 'font-size': '16px'}
        )
    ], style={'padding': '10px', 'background-color': '#e6e6e6'}),

    html.Hr(),

    # Data table
    html.Div([
        html.H3('Animal Data:'),
        dash_table.DataTable(
            id='datatable-id',
            columns=[
                {"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns
            ],
            data=df.to_dict('records'),
            editable=False,
            filter_action="native",
            sort_action="native",
            sort_mode="multi",
            column_selectable="single",
            row_selectable="single",
            row_deletable=False,
            selected_rows=[0],
            page_action="native",
            page_current=0,
            page_size=10,
            style_cell={
                'overflow': 'hidden',
                'textOverflow': 'ellipsis',
                'maxWidth': 0,
                'font-size': '12px'
            },
            style_header={
                'backgroundColor': 'rgb(30, 30, 30)',
                'color': 'white',
                'fontWeight': 'bold'
            },
            style_data={
                'whiteSpace': 'normal',
                'height': 'auto',
            }
        )
    ]),

    html.Hr(),

    # Charts section
    html.Div([
        # Left: Geolocation chart
        html.Div([
            html.H3('Animal Location:'),
            html.Div(id='map-id')
        ], style={'width': '50%', 'display': 'inline-block', 'vertical-align': 'top'}),

        # Right: Pie chart of breed distribution (as shown in example)
        html.Div([
            html.H3('Breed Distribution:'),
            dcc.Graph(id='breed-chart')
        ], style={'width': '50%', 'display': 'inline-block', 'vertical-align': 'top'})
    ])
])

# =========================
# Data Service Layer
# =========================

def build_query(filter_type):
    if filter_type == 'water':
        return {
            "animal_type": "Dog",
            "breed": {"$in": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'mountain':
        return {
            "animal_type": "Dog",
            "breed": {"$in": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
                              "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }
    elif filter_type == 'disaster':
        return {
            "animal_type": "Dog",
            "breed": {"$in": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                              "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }
    else:
        return {}


def get_filtered_data(filter_type):
    query = build_query(filter_type)
    results = shelter.read(query)
    return format_dataframe(results)


def format_dataframe(data):
    df = pd.DataFrame.from_records(data)
    if '_id' in df.columns:
        df.drop(columns=['_id'], inplace=True)
    return df

#############################################
# Interaction Between Components / Controller
#############################################

# Callback for filtering data based on the radio button selection
@app.callback(
    Output('datatable-id', 'data'),
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):
    df_filtered = get_filtered_data(filter_type)
    return df_filtered.to_dict('records')

# Callback for highlighting selected row in the data table
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_rows')]
)
def update_styles(selected_rows):
    if selected_rows is None or len(selected_rows) == 0:
        return []
    return [{
        'if': {'row_index': i},
        'background_color': '#D2F3FF',
        'color': 'black'
    } for i in selected_rows]

# Callback for updating the map based on selected row
@app.callback(
    Output('map-id', 'children'),
    [Input('datatable-id', 'derived_virtual_data'),
     Input('datatable-id', 'derived_virtual_selected_rows')]
)
def update_map(viewData, selectedRows):
    # Handle case when no data is available
    if viewData is None or len(viewData) == 0:
        return dl.Map(style={'width': '100%', 'height': '500px'},
                    center=[30.75, -97.48], zoom=10,
                    children=[dl.TileLayer(id="base-layer-id")])

    dff = pd.DataFrame.from_dict(viewData)

    # Determine which row to display on the map
    if selectedRows is None or len(selectedRows) == 0:
        row = 0
    else:
        row = selectedRows[0]

    # For safety, ensure we have data and row is valid
    if dff.empty or row >= len(dff):
        return dl.Map(style={'width': '100%', 'height': '500px'},
                    center=[30.75, -97.48], zoom=10,
                    children=[dl.TileLayer(id="base-layer-id")])

    # Find the columns containing latitude and longitude
    # This assumes your data has lat/long columns - adjust as needed
    lat_col = next((col for col in dff.columns if 'lat' in col.lower()), None)
    lon_col = next((col for col in dff.columns if 'lon' in col.lower() or 'lng' in col.lower()), None)

    # Find columns for animal information
    breed_col = next((col for col in dff.columns if 'breed' in col.lower()), None)
    name_col = next((col for col in dff.columns if 'name' in col.lower()), None)

    # Default to hardcoded column indices if we can't find the columns by name
    # These need to be adjusted based on your actual data structure
    try:
        if lat_col and lon_col and breed_col and name_col:
            lat = float(dff.iloc[row][lat_col])
            lon = float(dff.iloc[row][lon_col])
            breed = str(dff.iloc[row][breed_col])
            name = str(dff.iloc[row][name_col])
        else:
            # Adjust these indices based on your data structure
            lat = float(dff.iloc[row, 13])  # Assuming latitude is in column 13
            lon = float(dff.iloc[row, 14])  # Assuming longitude is in column 14
            breed = str(dff.iloc[row, 4])   # Assuming breed is in column 4
            name = str(dff.iloc[row, 9])    # Assuming name is in column 9
    except (IndexError, ValueError) as e:
        print(f"Error accessing data: {e}")
        # Default to Austin coordinates
        return dl.Map(style={'width': '100%', 'height': '500px'},
                    center=[30.75, -97.48], zoom=10,
                    children=[dl.TileLayer(id="base-layer-id")])

    # Return the map with the marker for the selected animal
    return dl.Map(
        style={'width': '100%', 'height': '500px'},
        center=[lat, lon], zoom=12,
        children=[
            dl.TileLayer(id="base-layer-id"),
            dl.Marker(
                position=[lat, lon],
                children=[
                    dl.Tooltip(breed),
                    dl.Popup([
                        html.H4("Animal Information"),
                        html.P(f"Name: {name}"),
                        html.P(f"Breed: {breed}")
                    ])
                ]
            )
        ]
    )

# Callback for updating the breed distribution chart as a pie chart (as per example in requirements)
@app.callback(
    Output('breed-chart', 'figure'),
    [Input('datatable-id', 'derived_virtual_data')]
)
def update_breed_chart(viewData):
    if viewData is None or len(viewData) == 0:
        # Return empty figure if no data
        return px.pie(title="No data available")

    # Convert to DataFrame
    dff = pd.DataFrame.from_dict(viewData)

    # Find the breed column
    breed_col = next((col for col in dff.columns if 'breed' in col.lower()), 'breed')

    # Count occurrences of each breed
    breed_counts = dff[breed_col].value_counts().reset_index()
    breed_counts.columns = ['Breed', 'Count']

    # Limit to top breeds for readability (adjust as needed)
    top_n = min(8, len(breed_counts))  # Show at most 8 breeds
    breed_counts = breed_counts.head(top_n)

    # Create the pie chart as shown in the example
    fig = px.pie(
        breed_counts,
        values='Count',
        names='Breed',
        title='Breed Distribution in Current Selection',
        color_discrete_sequence=px.colors.qualitative.Set3
    )

    # Update layout for better readability
    fig.update_traces(textposition='inside', textinfo='percent+label')
    fig.update_layout(
        margin=dict(l=20, r=20, t=40, b=20),
        height=500,
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=-0.3,
            xanchor="center",
            x=0.5
        )
    )

    return fig

print("REACHED RUN SERVER")
# Run the app with a specific port to avoid conflicts
app.run(port=8051, debug=True, use_reloader=False)

['admin', 'config', 'local']
REACHED RUN SERVER
